# Nghiệm thu tích hợp Milestone 2 trên Colab CPU

TV1 chạy module thật TV2–TV5, kiểm tra đầu ra và bàn giao bằng chứng cho TV5. Notebook không viết báo cáo MS2.

Chọn **Runtime → Change runtime type → CPU**. Đầu vào phải là snapshot MS1 đã dừng ghi, có discovery, checkpoint download/extract, raw, corpus và các sidecar. ZIP log/benchmark không thay thế snapshot. Mỗi thành viên dùng output riêng; không gộp các manifest bằng cách chép đè.

Chạy mẫu trước bằng danh sách ID. Chỉ đặt `ids=None` khi snapshot chứa toàn bộ phạm vi thành viên. Notebook ghi rõ mẫu số được giao và số bài thực sự chạy; số liệu local/mẫu không đại diện toàn corpus.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Chọn mã nguồn và phạm vi

Đặt `CODE_REF` là commit đã review hoặc nhánh tích hợp. Notebook ghi lại SHA thực tế. Sửa đường dẫn snapshot và output cho đúng tài khoản Drive. Giữ nguyên `RUN_TAG` khi tiếp tục checkpoint; dùng tag mới để đo một lượt chạy mới. Đổi mã nguồn phải khởi động lại runtime trước khi import lại module.

Để chạy toàn nhóm, khai báo đủ tv1–tv5 trong `JOBS`, mỗi mục có snapshot riêng. `total_assigned` là số ID được giao cho thành viên, kể cả ID không có bài; kiểm tra lại theo phân công nhóm.

In [ ]:
from pathlib import Path
CODE_REF = 'feat/m2-tv1-integration'
RUN_TAG = 'ms2-acceptance-20261110'
OUTPUT_BASE = Path('/content/drive/MyDrive/arxiv_ms2_acceptance') / RUN_TAG
JOBS = [
    {
        'member': 'tv1',
        'input': '/content/drive/MyDrive/arxiv_ms1_snapshots/tv1/data',
        'ids': ['2508.17481', '2508.17740', '2508.17871'],
        'total_assigned': 5000,
        'assignment_range': '2508.12872:2508.17871',
    },
]
RESUME = True
CHECK_RESUME = True

In [ ]:
import subprocess, sys
REPO = Path('/content/arxiv-ms2-integration')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/LeoTkiet/arxiv-data-science.git', str(REPO)], check=True)
if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Checkout has local changes; keep them before changing CODE_REF.')
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', CODE_REF], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
CODE_SHA = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO) + '[reporting]'], check=True)
# Add the checked-out source even when this runtime previously installed another checkout.
sys.path.insert(0, str(REPO / 'src'))
print('Commit:', CODE_SHA)

## Chạy và lưu bằng chứng

Mỗi output có `acceptance.json`, số đo của lượt chạy, kết quả validation, checkpoint và các bảng. `complete` chỉ nói về phạm vi ID đã chọn. `full_assignment_accepted` còn yêu cầu không lọc ID, xử lý đủ số ID được giao và không thiếu discovery. Chỉ kết luận toàn nhóm sau khi đủ năm output và xử lý các lỗi còn lại.

Các cảnh báo xung đột BibTeX cần TV4/TV5 đối chiếu nội dung. Nếu có lỗi, đọc stage và artifact liên quan; không sửa trạng thái thành success. Kết quả resume được lưu riêng để không thay số đo của lượt parsing bằng thời gian đọc cache.

In [ ]:
import json
from collections import Counter
from arxiv_pipeline.common.utils import atomic_json
from arxiv_pipeline.main import expand_range
from arxiv_pipeline.pipeline import parse
from arxiv_pipeline.export.benchmark import execute_pipeline_benchmark

members = [job['member'] for job in JOBS]
if not JOBS or len(set(members)) != len(members):
    raise ValueError('JOBS must contain distinct member names.')
for job in JOBS:
    if job['member'] not in {'tv1', 'tv2', 'tv3', 'tv4', 'tv5'}:
        raise ValueError('Unknown member name.')
    if type(job['total_assigned']) is not int or job['total_assigned'] < 1:
        raise ValueError('total_assigned must be a positive integer.')
    if job['ids'] is not None and (not isinstance(job['ids'], list) or not job['ids']):
        raise ValueError('Use a nonempty ID list for a sample, or None for all records.')
    assigned = set(expand_range(job['assignment_range']))
    if len(assigned) != job['total_assigned']:
        raise ValueError('Assignment range and denominator differ.')
    if job['ids'] is not None and not set(job['ids']).issubset(assigned):
        raise ValueError('Selected sample is outside the assigned range.')
    parse.validate_roots(job['input'], OUTPUT_BASE / job['member'])
for job in JOBS:
    for other in JOBS:
        parse.validate_roots(job['input'], OUTPUT_BASE / other['member'])

results = []
for job in JOBS:
    output = OUTPUT_BASE / job['member']
    output.mkdir(parents=True, exist_ok=True)
    atomic_json(output / 'acceptance-config.json', dict(job=job, code_sha=CODE_SHA, resume=RESUME))
    benchmark_error = None
    try:
        execute_pipeline_benchmark(
            job['input'], output, identifiers=job['ids'], resume=RESUME,
            total_assigned=job['total_assigned'],
            summary_out=output / 'summaries/parsing_summary.json')
    except Exception as exc:
        # Leave the stage evidence available for diagnosis; do not reuse old metrics.
        benchmark_error = f'{type(exc).__name__}: {exc}'
        atomic_json(output / 'summaries/benchmark-error.json', dict(error=benchmark_error))
    if benchmark_error:
        result = dict(member=job['member'], code_sha=CODE_SHA, complete=False,
                      full_assignment_accepted=False, benchmark_error=benchmark_error)
        atomic_json(output / 'acceptance.json', result)
        results.append(result)
        print(job['member'], result)
        continue

    summary = json.loads((output / 'parsing_summary.json').read_text())
    stages = json.loads((output / 'manifests/stages.json').read_text())
    atomic_json(output / 'summaries/parse-run.json', summary)
    issues = []
    for stage in stages:
        if stage['status'] not in parse.TERMINAL:
            issues.append(dict(stage=stage['stage'], paper_id=stage['paper_id'],
                               version=stage['version'], status=stage['status'], error=stage.get('error')))
        if stage['stage'] == 'validate_parse' and stage.get('artifacts', {}).get('validation'):
            report = json.loads((output / stage['artifacts']['validation']).read_text())
            issues.extend(dict(stage='validate_parse', **issue) for issue in report['errors'] + report['warnings'])
    resume_events = []
    resume_ok = None
    if CHECK_RESUME:
        resumed = parse.run(job['ids'], output, input_root=Path(job['input']),
                            resume=True, observer=resume_events.append)
        atomic_json(output / 'summaries/resume-events.json', resume_events)
        resume_ok = bool(summary['complete'] and resumed['complete'] and resume_events
                         and all(event['cached'] for event in resume_events))

    outcomes = summary['paper_results']
    excluded = summary['excluded_inputs']
    accounted = {p['arxiv_id'] for p in outcomes} | {
        p.get('arxiv_id') or p.get('input_id') for p in excluded}
    assigned = set(expand_range(job['assignment_range']))
    result = dict(
        member=job['member'], code_sha=CODE_SHA,
        scope='all_discovered_records' if job['ids'] is None else 'selected_sample',
        selected_ids=job['ids'], total_assigned=job['total_assigned'],
        attempted_papers=len(outcomes), accounted_inputs=len(accounted),
        unseen_assigned_ids=sorted(assigned - accounted),
        out_of_range_ids=sorted(accounted - assigned),
        complete=summary['complete'], input_errors=summary['input_errors'],
        paper_statuses=dict(Counter(p['status'] for p in outcomes)),
        version_statuses=dict(Counter(v['status'] for p in outcomes for v in p['versions'])),
        incomplete_papers=[p['arxiv_id'] for p in outcomes if not p['complete']],
        resume_all_cached=resume_ok, issues=issues,
        full_assignment_accepted=bool(
            job['ids'] is None and summary['complete'] and not summary['input_errors']
            and accounted == assigned
            and (not CHECK_RESUME or resume_ok)))
    atomic_json(output / 'acceptance.json', result)
    results.append(result)
    print(job['member'], {k: v for k, v in result.items() if k not in {'issues', 'selected_ids', 'unseen_assigned_ids', 'out_of_range_ids', 'incomplete_papers'}})

atomic_json(OUTPUT_BASE / 'group-acceptance.json', dict(
    code_sha=CODE_SHA, members=results,
    complete=bool(set(members) == {'tv1', 'tv2', 'tv3', 'tv4', 'tv5'}
                  and all(r['full_assignment_accepted'] for r in results))))
print('Evidence:', OUTPUT_BASE)

## Bàn giao

Giữ toàn bộ output theo thành viên, đặc biệt `acceptance-config.json`, `acceptance.json`, `group-acceptance.json`, `summaries/`, `state/`, `manifests/`, `validation/`, `parsed/` và `tables/`. Không chỉ gửi ZIP log MS1 hoặc ba bảng rời.

TV1 xử lý Resolver/Orchestrator, TV2 xử lý cấu trúc cây, TV3 xử lý chuẩn hóa nội dung, TV4 xử lý Bibliography/Deduplication/citation, TV5 xử lý QA, thống kê và báo cáo. TV5 dùng số liệu đo ở đây để viết báo cáo MS2, kèm phạm vi và môi trường thực tế.